# Model Validation Analysis

Human-annotated evaluation of 46 questions across 6 LLM × embedding combinations.

- **T** = Correct (1.0)
- **PC** = Partially Correct (0.5)
- **F** = Wrong (0.0)

Difficulty tiers: Easy (Q1–Q17), Medium (Q18–Q31), Hard (Q32–Q46)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

df = pd.read_csv('../../Model Validation - Sheet1.csv', index_col=0)
df = df.drop(columns=['Question', 'Notes'], errors='ignore')
df.index = [f'Q{i+1}' for i in range(len(df))]

MODELS = [
    'Claude + Ollama Embeddings',
    'Claude + Sentence Embeddings',
    'Claude + OpenAI Embeddings',
    'Ollama + Ollama Embeddings',
    'Ollama + Sentence Embeddings',
    'Ollama + OpenAI Embeddings',
]
df.columns = MODELS

SCORE_MAP = {'T': 1.0, 'PC': 0.5, 'F': 0.0}
scores = df.applymap(lambda x: SCORE_MAP.get(str(x).strip(), np.nan))

EASY   = [f'Q{i}' for i in range(1, 18)]
MEDIUM = [f'Q{i}' for i in range(18, 32)]
HARD   = [f'Q{i}' for i in range(32, 47)]

print('Data loaded:', scores.shape)

## Overall Accuracy per Model

In [ ]:
overall = scores.mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2196F3' if 'Claude' in m else '#FF9800' for m in overall.index]
bars = ax.bar(range(len(overall)), overall.values, color=colors, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, overall.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(range(len(overall)))
short_labels = [m.replace('Embeddings', 'Emb.') for m in overall.index]
ax.set_xticklabels(short_labels, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Mean Score (T=1, PC=0.5, F=0)')
ax.set_ylim(0, 1.1)
ax.set_title('Overall Accuracy by Model Combination')
ax.axhline(0.75, color='gray', linestyle='--', linewidth=0.8, label='0.75 threshold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('overall_accuracy.png', dpi=150)
plt.show()
print(overall.to_string())

## Accuracy by Difficulty Tier

In [ ]:
tier_scores = pd.DataFrame({
    'Easy':   scores.loc[EASY].mean(),
    'Medium': scores.loc[MEDIUM].mean(),
    'Hard':   scores.loc[HARD].mean(),
})

short = [m.replace(' Embeddings', '').replace(' + ', '\n') for m in tier_scores.index]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(tier_scores))
w = 0.25
for i, (tier, color) in enumerate(zip(['Easy', 'Medium', 'Hard'], ['#4CAF50', '#2196F3', '#F44336'])):
    bars = ax.bar(x + i*w, tier_scores[tier], w, label=tier, color=color,
                  edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, tier_scores[tier]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + w)
ax.set_xticklabels(short, fontsize=8)
ax.set_ylabel('Mean Score')
ax.set_ylim(0, 1.15)
ax.set_title('Accuracy by Difficulty Tier')
ax.legend()
plt.tight_layout()
plt.savefig('accuracy_by_tier.png', dpi=150)
plt.show()
print(tier_scores.round(3).to_string())

## T / PC / F Breakdown per Model

In [ ]:
counts = {}
for col in MODELS:
    vc = df[col].str.strip().value_counts()
    counts[col] = {'T': vc.get('T', 0), 'PC': vc.get('PC', 0), 'F': vc.get('F', 0)}
counts_df = pd.DataFrame(counts).T

short = [m.replace(' Embeddings', '').replace(' + ', '\n') for m in counts_df.index]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(counts_df))
w = 0.25
for i, (label, color) in enumerate(zip(['T', 'PC', 'F'], ['#4CAF50', '#FFC107', '#F44336'])):
    ax.bar(x + i*w, counts_df[label], w, label=label, color=color,
           edgecolor='black', linewidth=0.5)

ax.set_xticks(x + w)
ax.set_xticklabels(short, fontsize=8)
ax.set_ylabel('Count')
ax.set_title('Correct / Partial / Wrong Count per Model')
ax.legend(title='Label')
plt.tight_layout()
plt.savefig('tpcf_counts.png', dpi=150)
plt.show()
print(counts_df.to_string())

## Heatmap: Score per Question per Model

In [ ]:
fig, ax = plt.subplots(figsize=(10, 14))
cmap = mcolors.LinearSegmentedColormap.from_list('tfpc', ['#F44336', '#FFC107', '#4CAF50'])
im = ax.imshow(scores.values, aspect='auto', cmap=cmap, vmin=0, vmax=1)

short_cols = [m.replace(' Embeddings', '').replace(' + ', '\n') for m in MODELS]
ax.set_xticks(range(len(MODELS)))
ax.set_xticklabels(short_cols, fontsize=8)
ax.set_yticks(range(len(scores)))
ax.set_yticklabels(scores.index, fontsize=7)

for sep in [16.5, 30.5]:
    ax.axhline(sep, color='black', linewidth=1.5)

for i in range(len(scores)):
    for j in range(len(MODELS)):
        val = scores.values[i, j]
        label = {1.0: 'T', 0.5: 'PC', 0.0: 'F'}.get(val, '')
        ax.text(j, i, label, ha='center', va='center', fontsize=6,
                color='white' if val < 0.4 else 'black')

plt.colorbar(im, ax=ax, label='Score')
ax.set_title('Per-Question Scores (Easy | Medium | Hard)')
plt.tight_layout()
plt.savefig('heatmap.png', dpi=150)
plt.show()

## Claude vs Ollama Summary

In [ ]:
claude_cols = [m for m in MODELS if 'Claude' in m]
ollama_cols = [m for m in MODELS if 'Ollama' in m and 'Claude' not in m]

claude_mean = scores[claude_cols].values.mean()
ollama_mean = scores[ollama_cols].values.mean()

print(f'Claude average (all embeddings): {claude_mean:.3f}')
print(f'Ollama average (all embeddings): {ollama_mean:.3f}')
print()
print('Per embedding comparison:')
for emb in ['Ollama Embeddings', 'Sentence Embeddings', 'OpenAI Embeddings']:
    c = scores[f'Claude + {emb}'].mean()
    o = scores[f'Ollama + {emb}'].mean()
    print(f'  {emb:<25}  Claude={c:.3f}  Ollama={o:.3f}  diff={c-o:+.3f}')